<div class="alert alert-info">
    <strong>导航</strong>
    <ul>
        <li>上一节： <a href="9_18_scientific_measurement_from_images.ipynb">9.18 从图像到科学量</a></li>
        <li>下一节： <a href="9_x_further_reading_and_workflow.ipynb">9.x 延伸阅读与后续实践方向</a></li>
    </ul>
</div>


## 9.19 谱线物理量与运动学解释：速度约定、Moment、柱密度与 PV 图

第 9.7 节已经说明了如何从观测数据得到连续谱扣除后的谱线 cube，并生成 channel map、moment map、PV 图和全局谱线。第 9.13 节进一步引入三维 mask、谱线源搜索和简化运动学拟合。本节把这些图像产品再向前推进一步：谱线 cube 中每一个通道的强度、每一张 moment 图和每一条全局线轮廓，如何转化为速度、柱密度、气体质量和旋转信息。

谱线分析比连续谱测量多出一个关键维度。连续谱图像主要关心角位置、通量密度和形态；谱线 cube 还携带频率或速度坐标。这个坐标不是简单的横轴标签，而是由静止频率、观测频率、速度定义和参考系共同决定。若这些约定没有写清，同一组数据可能给出略有不同的速度中心、线宽和动力学解释。对于银河系 H I、近邻星系 CO、高红移分子线或吸收线系统，这种差异的量级和科学影响并不相同，因此需要把约定本身纳入结果的一部分。


### 9.19.1 从频率到速度：坐标变换不是中性操作

谱线观测首先测得的是频率。若谱线静止频率为 $\nu_0$，观测频率为 $\nu$，则频移可以用不同速度约定表达。射电约定、光学约定和相对论约定常写为

$$
v_{\rm radio}=c\left(1-\frac{\nu}{\nu_0}\right),
$$

$$
v_{\rm optical}=c\left(\frac{\nu_0}{\nu}-1\right),
$$

$$
v_{\rm rel}=c\,\frac{\left(\nu_0/\nu\right)^2-1}{\left(\nu_0/\nu\right)^2+1}.
$$

在低速极限下，三者都趋近于 $v/c\simeq (\nu_0-\nu)/\nu_0$。这也是许多近邻天体处理流程看起来不太区分速度约定的原因。但这种近似并不意味着约定可以省略。速度达到几千到数万 $\mathrm{km\,s^{-1}}$ 时，不同约定之间的差别会进入若干十到上千 $\mathrm{km\,s^{-1}}$ 的量级；即使在低速情形，若研究目标是窄线、精密系统速度或多个数据集的叠加，约定差异也可能成为系统误差。


![velocity conventions](figures/practical_line_velocity_conventions.png)

图中的横轴代表用相对论速度设定的同一组观测频率，纵轴代表在不同约定下报告的速度。低速区域三条曲线几乎重合，高速区域开始明显分离。这个图并不表示某一种约定永远“正确”，而是强调速度数值依赖定义。射电数据文件、CASA image header、FITS header 和论文表格中通常会同时出现 rest frequency、spectral reference frame、velocity convention 或 spectral type 等信息；缺少这些信息时，谱线中心和线宽不能被完整复现。

参考系是另一个容易被低估的层次。频率可以被标到 topocentric、barycentric、LSRK、heliocentric 等参考系。地球自转、公转和太阳相对于本地标准静止系的运动都会改变投影速度。对于宽线星系整体轮廓，这些差别有时只表现为小的零点平移；对于窄线云、脉泽或多历元吸收线，它们会直接影响速度对齐。谱线 cube 的速度轴因此应被理解为“经过某一组约定定义后的坐标”，而不是从相关器输出自然长出来的物理真值。


### 9.19.2 Channel width、spectral resolution 与 Hanning smoothing

谱线 cube 的通道间隔通常来自频率网格。例如在射电速度近似下，频率间隔 $\Delta\nu$ 对应的速度间隔约为

$$
\Delta v \simeq c\,\frac{\Delta\nu}{\nu_0}.
$$

这个量常被称为 channel width，但它并不总等于谱分辨率。谱分辨率描述的是仪器和后处理对窄线的响应宽度，取决于相关器通道响应、加窗、平滑和重采样。Hanning smoothing 是常见例子：相邻通道按 $0.25,0.5,0.25$ 加权后，窄带振铃和 Gibbs 旁瓣会减弱，但有效谱分辨率变宽，相邻通道噪声也会变得相关。若之后没有抽取隔一通道的独立采样，谱线积分误差不能继续按独立通道数的平方根简单缩放。

这一区分在两个场景中特别重要。第一，窄线或吸收线的峰值会被谱响应稀释，线中心和峰深依赖通道化方式。第二，线宽校正需要同时考虑物理速度弥散、旋转投影、通道响应和数据处理造成的平滑。写出“channel width = 5 km/s”只能说明网格步长；若要让线宽测量可比较，还应说明谱响应或是否经过 Hanning smoothing、重采样、binning。


### 9.19.3 Moment 图：把三维信息压缩成二维量

谱线 cube 可写为 $I(x,y,v)$。最常见的三个 moment 定义为

$$
M_0(x,y)=\sum_i I(x,y,v_i)\,\Delta v,
$$

$$
M_1(x,y)=\frac{\sum_i v_i I(x,y,v_i)}{\sum_i I(x,y,v_i)},
$$

$$
M_2^2(x,y)=\frac{\sum_i [v_i-M_1(x,y)]^2 I(x,y,v_i)}{\sum_i I(x,y,v_i)}.
$$

$M_0$ 是积分强度，常被用于柱密度或积分通量；$M_1$ 是强度加权平均速度，常被称为速度场；$M_2$ 是强度加权速度宽度。需要注意的是，$M_1$ 并不是某个位置处气体的唯一速度，$M_2$ 也不等于纯粹的湍动速度弥散。只要视线方向存在多个速度分量、低信噪比通道、残余连续谱或 beam smearing，这些 moment 就会混合多种效应。

因此，moment 图的科学意义由 mask 决定。没有 mask 时，噪声会在大量空通道中积累，使 $M_0$ 的背景起伏变大，并使 $M_1$ 在低信噪比区域变得不稳定。过紧的 mask 会截掉低亮度线翼和弥散外延气体，导致积分通量和线宽偏低。较合理的 mask 通常结合空间平滑、谱向平滑、阈值、连通性和人工检查，使弱发射能够被保留，同时避免把随机噪声纳入积分。


![moment mask bias](figures/practical_line_moment_mask_bias.png)

上图用一条模拟谱线结构说明 mask 对 moment 的影响。无 mask 积分会把噪声沿谱向累计，积分强度曲线出现额外起伏，速度场在弱发射区域变得异常。过紧 mask 只保留峰值附近通道，线翼被截去，积分通量偏低，速度场也会向高信噪比的核心通道偏移。信号 mask 并不保证无偏，但它把“哪些体素被认为属于谱线发射”这一判断显式化，使误差讨论有了明确对象。

在真实分析中，moment 图最好与 channel map、PV 图和全局谱线一起阅读。若一个速度梯度只在 $M_1$ 图中出现，却不能在 channel map 中看到连续移动的发射结构，运动学解释就不稳固。若 $M_2$ 的高值出现在亮核附近，也需要区分真实速度弥散、多个速度分量、未分辨旋转和 beam smearing。moment 图是压缩摘要，而不是谱线 cube 的替代品。


### 9.19.4 从积分强度到柱密度和气体质量

谱线物理量的推导通常从单位转换开始。干涉成像常输出单位为 $\mathrm{Jy\,beam^{-1}}$ 的 cube。对于频率 $\nu$、合成波束主轴 $\theta_{\rm maj}$ 和 $\theta_{\rm min}$，Rayleigh-Jeans 近似下的亮温可写为

$$
T_{\rm B}[\mathrm{K}]\simeq 1.222\times 10^6\,
\frac{S_\nu[\mathrm{Jy\,beam^{-1}}]}{\nu_{\rm GHz}^2\,\theta_{\rm maj}[\mathrm{arcsec}] \theta_{\rm min}[\mathrm{arcsec}]}.
$$

对于光学薄的 H I 21 cm 发射，柱密度关系为

$$
N_{\rm HI}[\mathrm{cm^{-2}}]=1.823\times 10^{18}\int T_{\rm B}\,dv[\mathrm{K\,km\,s^{-1}}].
$$

这个公式的简洁性容易掩盖其假设：它要求发射近似光学薄，并且亮温积分已经与 beam、速度轴和 mask 一致。若 H I 存在显著自吸收或高光深，真实柱密度会被低估。对于分子线，常见做法不是直接从亮温得到总氢分子柱密度，而是通过 CO luminosity 和 $\alpha_{\rm CO}$ 或其他激发/丰度模型估计分子气体质量；这会引入金属丰度、激发条件、光深和多能级辐射转移等额外假设。


![column density and mass workflow](figures/practical_line_column_density_mass.png)

若目标是整体现象而非逐像素柱密度，常使用积分线通量估计 H I 质量。低红移近似下常见公式为

$$
M_{\rm HI}[M_\odot] \simeq 2.356\times 10^5\,\frac{D_{\rm Mpc}^2}{1+z}\,\left(\int S\,dv\right)[\mathrm{Jy\,km\,s^{-1}}].
$$

近邻星系中 $(1+z)$ 因子通常接近 1，但在统一写法中保留它有助于说明频率宽度、速度宽度和宇宙学距离之间的约定。误差预算至少包含热噪声、通道相关、mask 选择、通量标定、主波束校正、残余连续谱、缺短间距造成的 resolved-out flux，以及距离误差。若通量来自三维 mask，热噪声项可以近似写成

$$
\sigma_F \simeq \sigma_{\rm ch}\,\Delta v\,\sqrt{N_{\rm beam}\,N_{\rm chan,eff}},
$$

其中 $N_{\rm beam}$ 是独立 beam 数，$N_{\rm chan,eff}$ 是考虑谱向相关后的有效通道数。这个公式只是误差结构的入口；当 mask 由数据本身决定时，mask 的随机涨落与系统选择效应还会额外影响通量。对于弱源，改变平滑尺度或阈值后重复测量，常比只报告一个形式误差更能反映真实不确定度。


### 9.19.5 PV 图与 beam smearing：速度脊线不是旋转曲线本身

PV 图把 cube 沿某条空间路径切开，显示位置和速度的二维分布。沿星系主轴的 PV 图常用于估计旋转曲线，因为在理想薄盘、已知倾角、已知位置角且空间分辨率足够高的情况下，最大速度边界或亮度脊线可以追踪圆周运动。然而真实 PV 图并不只由旋转曲线决定。气体表面密度、速度弥散、盘厚、翘曲、非圆运动、beam、通道响应和切片宽度都会改变图上的亮度分布。

Beam smearing 是最常见的陷阱之一。若一个 beam 内覆盖了明显的速度梯度，观测到的谱线就是多个位置速度的加权叠加。中心区域尤其容易受影响，因为旋转曲线常在小半径处快速上升。结果是速度场看起来更平滑，中心线宽变大，PV ridge 低于真实最大旋转速度。用 moment 1 速度场直接拟合旋转曲线时，这种效应会把内区旋转速度压低；用 PV 图最高速度边界估计时，又会受到信噪比和阈值选择影响。


![beam smearing PV](figures/practical_line_beam_smearing_pv.png)

图中左侧为理想 PV 结构，虚线表示输入旋转曲线，亮色曲线表示沿每个位置取峰值得到的 PV ridge。右侧加入有限 beam 和谱响应后，中心速度梯度被抹平，峰值脊线偏离输入曲线，低亮度边界也发生变化。这个例子说明，PV 图中的“最高亮度位置”和“真实圆周速度”之间没有一一对应关系。

更稳健的运动学分析通常需要前向建模。模型先在三维空间中设定表面密度、旋转速度、速度弥散、倾角、位置角和系统速度，再经过投影、beam 卷积、谱响应和采样，最后与观测 cube 或 PV 图比较。这样做的好处是，空间分辨率和通道响应被放入模型，而不是在测量之后用简单修正项补救。对于低分辨率或低信噪比数据，模型参数之间可能高度退化；此时报告“哪些量被数据约束、哪些量依赖先验或固定假设”比给出形式上很精确的旋转曲线更重要。


### 9.19.6 线宽、倾角和动力学质量

全局谱线常用 $W_{20}$ 或 $W_{50}$ 表示在峰值 20% 或 50% 处的宽度。对于旋转支撑星系，线宽与投影旋转速度近似相关，常写成

$$
V_{\rm rot}\simeq \frac{W_{\rm corr}}{2\sin i},
$$

其中 $i$ 是盘面倾角，$W_{\rm corr}$ 是经过仪器分辨率、湍动弥散、红移和其他效应修正后的线宽。这个公式体现了谱线动力学的吸引力：一个未分辨星系的整体线轮廓就能提供旋转速度量级。但它也体现了主要风险：倾角小的近 face-on 系统中，$\sin i$ 很小，微小倾角误差会被放大；低质量星系中，随机运动和非圆运动所占比例更高，双角峰线型可能并不明显；相互作用或外流会使线宽不再代表平衡旋转。

若进一步估计动力学质量，常使用

$$
M_{\rm dyn}(<R)\simeq \frac{V_{\rm rot}^2 R}{G}.
$$

这里的 $R$ 必须与速度测量所代表的半径一致。若速度来自全局线宽而半径来自低阈值 H I 盘外缘，二者不一定对应同一动力学尺度。若速度来自 beam-smeared 的内区 PV ridge，而半径来自高分辨连续谱或光学图像，也可能混合不同成分。谱线动力学的关键不是把线宽代入公式，而是确认速度、半径、倾角和几何模型属于同一个物理系统。


![linewidth contributions](figures/practical_line_width_turbulence_inclination.png)

线宽图示强调了一个实用判断：观测线轮廓既包含旋转投影，也包含气体速度弥散、通道响应、beam smearing、mask 与信噪比选择。宽度修正不是纯粹的单位换算，而是模型假设。对于质量估计、Tully-Fisher 关系或星系气体动力学研究，线宽的定义、测量方法和修正策略必须随结果一起给出。


### 9.19.7 与真实软件工作流的对应

在 CASA 中，谱线 cube 的速度轴来自 `tclean`、`mstransform`、`cvel2` 或相关任务中的 rest frequency、outframe、veltype、start、width 等设置。CARTA、DS9 或 Python/FITS 工具读取 cube 时，也会根据 header 中的 spectral WCS 解释坐标。实际项目中，最常见的问题不是公式不会写，而是多个软件阶段对速度轴、通道顺序、参考系或 rest frequency 的理解不一致。处理日志应记录这些设置，尤其是在多观测项目拼接、不同谱线叠加或与文献速度比较时。

谱线 source finding 工具如 SoFiA 更接近三维 mask 和源检测层级，适合用于 H I cube 中的候选源、积分谱、可靠性评估和 moment 产品生成。PyBDSF 则主要面向连续谱图像的二维源表，不应直接替代谱线 cube 的三维检测逻辑。CARTA 适合承担交互式检查：逐通道查看发射连续性、比较不同 mask 下的 moment 图、抽取 PV 图和谱线。Python 生态中的 `spectral-cube`、`radio-beam`、`astropy.wcs` 等工具适合做可复现的单位转换、区域积分和误差传播。

一个完整的谱线结果至少应报告：目标谱线和 rest frequency；速度约定与参考系；cube 的 beam、像元、通道间隔和有效谱分辨率；是否 Hanning smoothing 或重采样；连续谱扣除方法；mask 构造方法；RMS 估计方式；积分通量、柱密度或质量公式及其假设；通量标定、距离、主波束、缺短间距和 mask 系统误差；若给出速度场、PV 图或旋转曲线，还应报告倾角、位置角、切片宽度、beam smearing 处理和模型退化性。


### 9.19.8 本节结论

谱线 cube 的科学解释是一条从坐标约定到物理模型的链条。频率必须通过速度约定和参考系转化为速度；通道间隔必须与有效谱分辨率区分；moment 图必须与 mask 和原始 cube 一起理解；柱密度和气体质量必须写清单位转换、光深或激发假设；PV 图和线宽必须放在空间分辨率、倾角和动力学模型中解释。

因此，谱线分析的目标不是尽快得到一组 moment 图，而是建立一套可追溯的解释路径：每一个物理量都能回到 cube 中的体素、mask、beam、速度轴和模型假设。做到这一点后，谱线数据才能从“漂亮的速度图”转化为可比较、可复现的气体物理和运动学测量。
